# ARC-AGI-3 Dual-Model Nine-Hour Runs

A normal Kaggle commit devotes its complete nine-hour GPU execution to all 25
official public games. A competition submission rerun has a separate fresh
nine-hour execution and devotes it to every unseen gateway game. Neither mode
uses an action cap; both reserve twenty minutes for model teardown and artifacts.

Notebook stdout and `dual-model-full-trace.jsonl` contain every model-emitted
response, plan/reasoning field, selected move, phase, and snapshot event. Hidden
internal tokens not emitted by a model are not accessible.


In [ ]:
import importlib.util
import json
import os
import subprocess
from pathlib import Path

os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
os.environ['VLLM_BLOCKSCALE_FP8_GEMM_FLASHINFER'] = '0'
os.environ['DUAL_ECHO_FULL_TRACE'] = '1'
os.environ['DUAL_TRACE_PATH'] = '/kaggle/working/dual-model-full-trace.jsonl'

arc_wheels = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
if importlib.util.find_spec('arc_agi') is None:
    subprocess.check_call([
        'python', '-m', 'pip', 'install', '--quiet', '--no-index',
        f'--find-links={arc_wheels}', 'arc-agi',
    ])

if importlib.util.find_spec('vllm') is None:
    wheels = list(Path('/kaggle/input').glob('**/vllm-0.23.0*.whl'))
    if not wheels:
        raise FileNotFoundError('vLLM 0.23.0 wheel missing from attached offline wheelhouse')
    wheelhouse = wheels[0].parent
    subprocess.check_call([
        'uv', 'pip', 'install', '--no-index', f'--find-links={wheelhouse}',
        'vllm==0.23.0', 'transformers==5.12.1',
    ])
print('Offline vLLM stack ready')

def locate_checkpoint(preferred, markers):
    preferred = Path(preferred)
    candidates = []
    if preferred.exists():
        candidates.append(preferred)
    for config in Path('/kaggle/input').glob('**/config.json'):
        lowered = str(config).lower()
        if any(marker in lowered for marker in markers):
            candidates.append(config.parent)
    for root in candidates:
        config = root / 'config.json'
        if config.exists():
            return root
        nested = sorted(root.glob('**/config.json'), key=lambda item: len(item.parts))
        if nested:
            return nested[0].parent
    raise FileNotFoundError(f'checkpoint not found for markers={markers}')

duck_model = locate_checkpoint(
    '/kaggle/input/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot',
    ('qwen3-6-27b', 'qwen3.6-27b', 'qwen'),
)
public_model = locate_checkpoint(
    '/kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1',
    ('gemma-4-31b', 'gemma4-31b'),
)
os.environ['ARC3_DUCK_MODEL_PATH'] = str(duck_model)
os.environ['ARC3_PUBLIC_MODEL_PATH'] = str(public_model)
print('Resolved Duck vision checkpoint:', duck_model)
print('Resolved public vision checkpoint:', public_model)


In [ ]:
import json
import sys
from pathlib import Path

PACKAGE = {'__init__.py': '"""Official Arcade test harness for the bidirectional handoff agent."""\n\nfrom .runner import ArcadeHarness, HarnessConfig, PolicyDecision, RunResult\nfrom .dual_runner import DualModelArcadeHarness\n\n__all__ = [\n    "ArcadeHarness", "DualModelArcadeHarness", "HarnessConfig",\n    "PolicyDecision", "RunResult",\n]\n', 'cli.py': 'from __future__ import annotations\n\nimport argparse\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\n\nfrom arc_agi import Arcade, OperationMode\n\nfrom .runner import ArcadeHarness, HarnessConfig\n\n\nSOURCE_URL = "https://github.com/BlockChain-BailBonds/arc3-milestone-handoff"\n\n\ndef parser() -> argparse.ArgumentParser:\n    out = argparse.ArgumentParser(description="Test an ARC-AGI-3 handoff through official Arcade")\n    out.add_argument("--games", help="comma-separated exact game IDs")\n    out.add_argument("--list-games", action="store_true")\n    out.add_argument("--attempts", type=int, default=2)\n    out.add_argument("--max-actions", type=int, default=200)\n    out.add_argument("--handoff-action", type=int, default=100)\n    out.add_argument(\n        "--preferred-direction",\n        choices=("duck_to_public", "public_to_duck"),\n        default="duck_to_public",\n    )\n    out.add_argument("--phase", choices=("practice", "two-stage"), default="practice")\n    out.add_argument("--output", type=Path, default=Path("artifacts/arcade-results.json"))\n    out.add_argument(\n        "--policy-adapter",\n        help="module:factory returning fresh (duck_policy, public_policy) instances",\n    )\n    out.add_argument(\n        "--confirm-live",\n        action="store_true",\n        help="required before actions are sent to arcprize.org",\n    )\n    return out\n\n\ndef main() -> None:\n    args = parser().parse_args()\n    if not os.getenv("ARC_API_KEY"):\n        raise SystemExit(\n            "ARC_API_KEY is required; get one from arcprize.org before discovery or live runs"\n        )\n    practice_arcade = Arcade(operation_mode=OperationMode.ONLINE)\n    if args.list_games:\n        empty = HarnessConfig(game_ids=("discovery",))\n        print(json.dumps(ArcadeHarness(empty, practice_arcade=practice_arcade).discover(), indent=2))\n        return\n    if not args.games:\n        raise SystemExit("--games is required unless --list-games is used")\n    if not args.confirm_live:\n        raise SystemExit("refusing to consume Arcade runs without --confirm-live")\n    if args.phase == "two-stage" and not args.policy_adapter:\n        raise SystemExit(\n            "two-stage competition refuses the built-in smoke policy; provide --policy-adapter"\n        )\n    config = HarnessConfig(\n        game_ids=tuple(item.strip() for item in args.games.split(",") if item.strip()),\n        attempts=args.attempts,\n        max_actions=args.max_actions,\n        handoff_action=args.handoff_action,\n        preferred_direction=args.preferred_direction,\n        source_url=SOURCE_URL,\n        output=args.output,\n    )\n    policies = {}\n    if args.policy_adapter:\n        module_name, separator, factory_name = args.policy_adapter.partition(":")\n        if not separator:\n            raise SystemExit("--policy-adapter must use module:factory syntax")\n        factory = getattr(importlib.import_module(module_name), factory_name)\n        duck_policy, public_policy = factory()\n        policies = {"duck_policy": duck_policy, "public_policy": public_policy}\n    harness = ArcadeHarness(config, practice_arcade=practice_arcade, **policies)\n    payload = harness.run_practice() if args.phase == "practice" else harness.run_two_stage()\n    print(json.dumps(payload, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'dual_cli.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\n\nfrom arc_agi import Arcade, OperationMode\n\nfrom .dual_runner import DualModelArcadeHarness\nfrom .model_runtime import VllmRuntime\nfrom .public_policies import create_public_dual_policies\nfrom .runner import HarnessConfig\nfrom .snapshot import SnapshotStore\n\n\nSOURCE_URL = "https://github.com/BlockChain-BailBonds/arc3-milestone-handoff"\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Public Qwen/Gemma snapshot handoff")\n    parser.add_argument("--games", required=True, help="comma-separated exact Arcade game IDs")\n    parser.add_argument(\n        "--max-actions", type=int, default=0,\n        help="test-only move cap; 0 means continue until Arcade WIN",\n    )\n    parser.add_argument("--handoff-action", type=int, default=100)\n    parser.add_argument("--qualification-levels", type=int, default=1)\n    parser.add_argument("--max-practice-rounds", type=int, default=3)\n    parser.add_argument("--preferred-direction", choices=("duck_to_public", "public_to_duck"), default="duck_to_public")\n    parser.add_argument("--artifacts", type=Path, default=Path("artifacts/dual-model"))\n    parser.add_argument("--confirm-live", action="store_true")\n    args = parser.parse_args()\n    if not os.getenv("ARC_API_KEY"):\n        raise SystemExit("ARC_API_KEY is required")\n    if not args.confirm_live:\n        raise SystemExit("refusing practice and competition scorecards without --confirm-live")\n    games = tuple(item.strip() for item in args.games.split(",") if item.strip())\n    config = HarnessConfig(\n        game_ids=games, attempts=2, max_actions=args.max_actions or None,\n        handoff_action=args.handoff_action,\n        qualification_levels=args.qualification_levels,\n        max_practice_rounds=args.max_practice_rounds,\n        preferred_direction=args.preferred_direction, source_url=SOURCE_URL,\n        output=args.artifacts / "results.json",\n    )\n    duck, public = create_public_dual_policies()\n    harness = DualModelArcadeHarness(\n        config,\n        practice_arcade=Arcade(operation_mode=OperationMode.ONLINE),\n        competition_arcade=Arcade(operation_mode=OperationMode.COMPETITION),\n        runtime=VllmRuntime(log_dir=args.artifacts / "vllm"),\n        duck_policy=duck, public_policy=public,\n        snapshot_store=SnapshotStore(args.artifacts / "snapshots"),\n    )\n    print(json.dumps(harness.run_two_stage(), indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'dual_runner.py': 'from __future__ import annotations\n\nimport json\nimport time\nimport uuid\nfrom dataclasses import asdict, dataclass, field, replace\nfrom pathlib import Path\nfrom typing import Any\n\nfrom arcengine import GameAction, GameState\n\nfrom .model_runtime import PUBLIC_MODEL_SPECS, ModelSpec, VllmRuntime\nfrom .online_learning import learn_transition\nfrom .phase_plan import Phase, competition_phase_plan, practice_phase_plan\nfrom .runner import HarnessConfig, Policy, PolicyContext, PolicyDecision, RunResult\nfrom .snapshot import AttemptSnapshot, SnapshotStore\n\n\n@dataclass\nclass _Session:\n    game_id: str\n    attempt: int\n    direction: str\n    env: Any\n    observation: Any\n    observations: list[Any]\n    memory: dict[str, Any] = field(default_factory=dict)\n    actions: int = 0\n    started: float = field(default_factory=time.monotonic)\n    snapshot_path: Path | None = None\n\n\nclass DualModelArcadeHarness:\n    """Keep Arcade sessions alive while swapping public model checkpoints."""\n\n    def __init__(\n        self,\n        config: HarnessConfig,\n        *,\n        practice_arcade: Any,\n        competition_arcade: Any,\n        runtime: VllmRuntime,\n        duck_policy: Policy,\n        public_policy: Policy,\n        snapshot_store: SnapshotStore,\n        model_specs: dict[str, ModelSpec] | None = None,\n        practice_game_ids: set[str] | None = None,\n    ) -> None:\n        config.validate()\n        self.config = config\n        self.practice_arcade = practice_arcade\n        self.competition_arcade = competition_arcade\n        self.runtime = runtime\n        self.policies = {"duck": duck_policy, "public": public_policy}\n        self.snapshots = snapshot_store\n        self.model_specs = model_specs or PUBLIC_MODEL_SPECS\n        self.practice_game_ids = practice_game_ids if practice_game_ids is not None else set(config.game_ids)\n        self.deadline: float | None = None\n\n    def _start_deadline(self) -> None:\n        self.deadline = (\n            time.monotonic() + self.config.global_time_limit_seconds\n            - self.config.shutdown_reserve_seconds\n        )\n\n    def _time_expired(self) -> bool:\n        return self.deadline is not None and time.monotonic() >= self.deadline\n\n    def run_two_stage(self) -> dict[str, Any]:\n        self._start_deadline()\n        practice = self._run_practice()\n        selected = practice["selected_directions"]\n        competition = self._run_competition(selected)\n        expected = len(self.config.game_ids)\n        if len(competition["runs"]) != expected:\n            raise RuntimeError(\n                f"competition cardinality violation: expected {expected} runs, "\n                f"recorded {len(competition[\'runs\'])}"\n            )\n        payload = {"practice": practice, "competition": competition}\n        output = self.config.output.with_name(\n            f"{self.config.output.stem}-dual-model{self.config.output.suffix}"\n        )\n        output.parent.mkdir(parents=True, exist_ok=True)\n        output.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n        return payload\n\n    def run_public_suite(self) -> dict[str, Any]:\n        """Use the full nine-hour execution only for the 25 public games."""\n        self._start_deadline()\n        return self._run_practice()\n\n    def run_private_competition(self) -> dict[str, Any]:\n        """Use a fresh nine-hour submission rerun for every unseen gateway game."""\n        self._start_deadline()\n        selected = {game: "duck_to_public" for game in self.config.game_ids}\n        return self._run_competition(selected)\n\n    def _open_sessions(self, arcade: Any, card_id: str, directions: dict[str, list[str]]) -> list[_Session]:\n        sessions: list[_Session] = []\n        for game_id, game_directions in directions.items():\n            for attempt, direction in enumerate(game_directions):\n                env = arcade.make(\n                    game_id, scorecard_id=card_id,\n                    save_recording=self.config.save_recordings, include_frame_data=True,\n                )\n                if env is None:\n                    raise RuntimeError(f"Arcade could not create {game_id}")\n                observation = env.observation_space\n                sessions.append(_Session(\n                    game_id, attempt, direction, env, observation, [observation]\n                ))\n        return sessions\n\n    def _run_practice(self) -> dict[str, Any]:\n        unknown = sorted(set(self.config.game_ids) - self.practice_game_ids)\n        pending = [game for game in self.config.game_ids if game in self.practice_game_ids]\n        all_results: list[RunResult] = []\n        rounds: list[dict[str, Any]] = []\n        for round_index in range(self.config.max_practice_rounds):\n            if not pending or self._time_expired():\n                break\n            round_payload = self._run_practice_round(tuple(pending), round_index)\n            rounds.append(round_payload)\n            all_results.extend(RunResult(**item) for item in round_payload["runs"])\n            selected_so_far = self._select(all_results)\n            best_levels = self._best_levels(all_results)\n            pending = [\n                game for game in self.config.game_ids\n                if best_levels.get(game, 0) < self.config.qualification_levels\n            ]\n            print(\n                f"PRACTICE_QUALIFICATION round={round_index + 1} "\n                f"qualified={len(self.config.game_ids) - len(pending)} "\n                f"pending={pending} selected={selected_so_far}"\n            )\n        unqualified = sorted(set(pending) | set(unknown))\n        selected = self._select(all_results)\n        for game in unqualified:\n            selected[game] = "duck_to_public"\n        if unqualified:\n            print(\n                "PRACTICE_UNQUALIFIED continuing_to_competition_with_duck_first="\n                f"{unqualified}"\n            )\n        ledger = self._improvement_ledger(all_results)\n        for game_id, entries in ledger.items():\n            for entry in entries:\n                print(f"IMPROVEMENT game={game_id} " + json.dumps(entry, sort_keys=True))\n        return {\n            "phase": "practice", "rounds": rounds,\n            "runs": [asdict(result) for result in all_results],\n            "selected_directions": selected,\n            "qualified_levels": self._best_levels(all_results),\n            "unqualified_games": unqualified,\n            "improvement_ledger": ledger,\n        }\n\n    def _run_practice_round(self, game_ids: tuple[str, ...], round_index: int) -> dict[str, Any]:\n        run_id = f"practice-r{round_index}-{uuid.uuid4().hex[:12]}"\n        card_id = self.practice_arcade.create_scorecard(\n            source_url=self.config.source_url,\n            tags=[*self.config.tags, "dual-model", "practice"],\n            opaque={"run_id": run_id, "round": round_index, "no_priors": True},\n        )\n        # The public first-place Duck/Qwen policy always gets the first attempt.\n        directions = {game: ["duck_to_public", "public_to_duck"] for game in game_ids}\n        sessions = self._open_sessions(self.practice_arcade, card_id, directions)\n        scorecard = None\n        try:\n            self._execute_phases(sessions, practice_phase_plan(), run_id)\n        finally:\n            self.runtime.stop()\n            scorecard = self.practice_arcade.close_scorecard(scorecard_id=card_id)\n        results = self._attach_official_scores(\n            [self._result(session) for session in sessions], scorecard\n        )\n        return self._payload("practice", card_id, scorecard, results, self._select(results))\n\n    def _run_competition(self, selected: dict[str, str]) -> dict[str, Any]:\n        expected_games = set(self.config.game_ids)\n        if set(selected) != expected_games:\n            missing = sorted(expected_games - set(selected))\n            extra = sorted(set(selected) - expected_games)\n            raise ValueError(f"competition selection mismatch: missing={missing}, extra={extra}")\n        run_id = f"competition-{uuid.uuid4().hex[:12]}"\n        card_id = self.competition_arcade.create_scorecard(\n            source_url=self.config.source_url,\n            tags=[*self.config.tags, "dual-model", "competition"],\n            opaque={"run_id": run_id, "no_priors": True, "directions": selected},\n        )\n        # Exactly one environment instance per game in competition mode.\n        sessions = self._open_sessions(\n            self.competition_arcade, card_id,\n            {game: [selected[game]] for game in self.config.game_ids},\n        )\n        if len(sessions) != len(self.config.game_ids):\n            raise RuntimeError("competition must create exactly one environment per input game")\n        scorecard = None\n        try:\n            self._execute_phases(sessions, competition_phase_plan(selected), run_id)\n        finally:\n            self.runtime.stop()\n            scorecard = self.competition_arcade.close_scorecard(scorecard_id=card_id)\n        results = [self._result(session) for session in sessions]\n        return self._payload("competition", card_id, scorecard, results, selected)\n\n    def _execute_phases(self, sessions: list[_Session], phases: tuple[Phase, ...], run_id: str) -> None:\n        for phase in phases:\n            if self._time_expired():\n                print(f"GLOBAL_TIME_LIMIT phase_skipped={phase}")\n                break\n            matching = [session for session in sessions if session.direction == phase.direction]\n            if not matching:\n                continue\n            print(\n                f"PHASE_START run={run_id} model={phase.model} "\n                f"direction={phase.direction} segment={phase.segment} games={len(matching)}"\n            )\n            self.runtime.activate(self.model_specs[phase.model])\n            for session_index, session in enumerate(matching):\n                if self._time_expired():\n                    break\n                if phase.segment == "second":\n                    if session.snapshot_path is None:\n                        raise RuntimeError("second segment has no first-segment snapshot")\n                    loaded = self.snapshots.load(\n                        session.snapshot_path, run_id=run_id,\n                        attempt_id=f"{session.game_id}-{session.attempt}",\n                    )\n                    session.memory = dict(loaded.world_model)\n                target: int | None = (\n                    self.config.handoff_action\n                    if phase.segment == "first" else self.config.max_actions\n                )\n                remaining_sessions = max(1, len(matching) - session_index)\n                remaining_seconds = max(0.0, (self.deadline or time.monotonic()) - time.monotonic())\n                session_deadline = time.monotonic() + remaining_seconds / remaining_sessions\n                self._advance(session, phase.model, target, session_deadline)\n                session.snapshot_path = self.snapshots.save(self._snapshot(session, run_id))\n                print(\n                    f"SNAPSHOT game={session.game_id} attempt={session.attempt} "\n                    f"step={session.actions} path={session.snapshot_path}"\n                )\n\n    def _advance(\n        self, session: _Session, model: str, target: int | None,\n        session_deadline: float | None = None,\n    ) -> None:\n        policy = self.policies[model]\n        while target is None or session.actions < target:\n            if self._time_expired() or (\n                session_deadline is not None and time.monotonic() >= session_deadline\n            ):\n                print(f"TIME_SLICE_END game={session.game_id} actions={session.actions}")\n                break\n            state = getattr(session.observation, "state", None)\n            if state == GameState.WIN:\n                break\n            if state in {GameState.NOT_PLAYED, GameState.GAME_OVER}:\n                decision = PolicyDecision(GameAction.RESET, reasoning="Arcade state requires reset")\n            else:\n                decision = policy.decide(session.observation, PolicyContext(\n                    game_id=session.game_id, direction=session.direction,\n                    active_policy=model, action_index=session.actions,\n                    handoff_action=self.config.handoff_action,\n                    observations=session.observations, memory=session.memory,\n                ))\n            reasoning = {"text": decision.reasoning} if decision.reasoning else None\n            observation = session.env.step(decision.action, data=decision.data, reasoning=reasoning)\n            if observation is None:\n                raise RuntimeError(f"Arcade returned no observation for {session.game_id}")\n            learn_transition(session.memory, session.observation, decision.action, observation)\n            session.observation = observation\n            session.observations.append(observation)\n            session.actions += 1\n\n    def _snapshot(self, session: _Session, run_id: str) -> AttemptSnapshot:\n        frame = getattr(session.observation, "frame", None)\n        if hasattr(frame, "tolist"):\n            frame = frame.tolist()\n        frame = frame or []\n        return AttemptSnapshot(\n            run_id=run_id, game_id=session.game_id,\n            attempt_id=f"{session.game_id}-{session.attempt}",\n            direction=session.direction, action_index=session.actions,\n            handoff_action=self.config.handoff_action,\n            levels_completed=int(getattr(session.observation, "levels_completed", 0)),\n            current_frame=frame, world_model=dict(session.memory),\n        )\n\n    @staticmethod\n    def _select(results: list[RunResult]) -> dict[str, str]:\n        selected: dict[str, RunResult] = {}\n        for result in results:\n            current = selected.get(result.game_id)\n            if current is None or DualModelArcadeHarness._metric(result) > DualModelArcadeHarness._metric(current):\n                selected[result.game_id] = result\n        return {game: result.direction for game, result in selected.items()}\n\n    @staticmethod\n    def _metric(result: RunResult) -> tuple[float, int, int]:\n        efficiency = -result.actions if result.levels_completed > 0 else -10**12\n        return (result.score, result.levels_completed, efficiency)\n\n    @classmethod\n    def _improvement_ledger(cls, results: list[RunResult]) -> dict[str, list[dict[str, Any]]]:\n        ledger: dict[str, list[dict[str, Any]]] = {}\n        best: dict[str, tuple[float, int, int]] = {}\n        for result in results:\n            metric = cls._metric(result)\n            previous = best.get(result.game_id, (0.0, 0, -10**12))\n            improved = metric > previous\n            if improved:\n                best[result.game_id] = metric\n            ledger.setdefault(result.game_id, []).append({\n                "attempt": result.attempt, "direction": result.direction,\n                "score": result.score, "levels_completed": result.levels_completed,\n                "actions": result.actions, "improved": improved,\n                "accepted_as_best": improved,\n            })\n        return ledger\n\n    @staticmethod\n    def _attach_official_scores(results: list[RunResult], scorecard: Any) -> list[RunResult]:\n        if scorecard is None:\n            return results\n        dumped = scorecard.model_dump(mode="json")\n        queues = {\n            environment.get("id"): list(environment.get("runs") or [])\n            for environment in dumped.get("environments") or []\n        }\n        scored: list[RunResult] = []\n        for result in results:\n            run = queues.get(result.game_id, [])\n            official = run.pop(0) if run else {}\n            scored.append(replace(result, score=float(official.get("score") or 0.0)))\n        return scored\n\n    @staticmethod\n    def _best_levels(results: list[RunResult]) -> dict[str, int]:\n        best: dict[str, int] = {}\n        for result in results:\n            best[result.game_id] = max(best.get(result.game_id, 0), result.levels_completed)\n        return best\n\n    @staticmethod\n    def _result(session: _Session) -> RunResult:\n        return RunResult(\n            game_id=session.game_id, attempt=session.attempt, direction=session.direction,\n            actions=session.actions,\n            levels_completed=int(getattr(session.observation, "levels_completed", 0)),\n            final_state=getattr(getattr(session.observation, "state", None), "name", "UNKNOWN"),\n            elapsed_seconds=round(time.monotonic() - session.started, 3),\n            handoff_reached=session.actions > 0,\n            score=0.0,\n        )\n\n    @staticmethod\n    def _payload(phase: str, card_id: str, scorecard: Any, results: list[RunResult], selected: dict[str, str]) -> dict[str, Any]:\n        return {\n            "phase": phase, "scorecard_id": card_id,\n            "scorecard": scorecard.model_dump(mode="json") if scorecard else None,\n            "runs": [asdict(result) for result in results],\n            "selected_directions": selected,\n        }\n', 'model_runtime.py': 'from __future__ import annotations\n\nimport json\nimport os\nimport subprocess\nimport time\nimport urllib.request\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass ModelSpec:\n    policy: str\n    served_name: str\n    model_path: Path\n    port: int = 8000\n    max_model_len: int = 32768\n    gpu_memory_utilization: float = 0.94\n    extra_args: tuple[str, ...] = field(default_factory=tuple)\n\n\nPUBLIC_MODEL_SPECS = {\n    "duck": ModelSpec(\n        policy="duck",\n        served_name="duck-qwen36-27b",\n        model_path=Path(os.getenv(\n            "ARC3_DUCK_MODEL_PATH",\n            "/kaggle/input/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot",\n        )),\n        # The RTX Pro 6000 is SM120. vLLM 0.23\'s automatic FP8 selection can\n        # choose FlashInfer\'s SM120 JIT path even when the offline wheelhouse\n        # lacks an nvcc architecture target for it. PyTorch scaled_mm supports\n        # the checkpoint without that JIT dependency.\n        extra_args=(\n            "--linear-backend", "torch",\n            "--tool-call-parser", "qwen3_coder",\n            "--reasoning-parser", "qwen3",\n        ),\n    ),\n    "public": ModelSpec(\n        policy="public",\n        served_name="public-gemma4-31b",\n        model_path=Path(os.getenv(\n            "ARC3_PUBLIC_MODEL_PATH",\n            "/kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1",\n        )),\n        extra_args=("--limit-mm-per-prompt", \'{"image":4}\'),\n    ),\n}\n\n\nclass VllmRuntime:\n    """Own exactly one local vLLM server and swap checkpoints explicitly."""\n\n    def __init__(self, *, log_dir: Path = Path("artifacts/vllm"), startup_timeout: int = 1200) -> None:\n        self.log_dir = log_dir\n        self.startup_timeout = startup_timeout\n        self.process: subprocess.Popen[bytes] | None = None\n        self.log_handle = None\n        self.active: ModelSpec | None = None\n\n    def activate(self, spec: ModelSpec) -> None:\n        if self.active == spec and self.process and self.process.poll() is None:\n            return\n        self.stop()\n        model_path = self._resolve_model_path(spec.model_path)\n        if not model_path.exists():\n            raise FileNotFoundError(f"missing attached public model snapshot: {spec.model_path}")\n        self.log_dir.mkdir(parents=True, exist_ok=True)\n        log_path = self.log_dir / f"{spec.policy}.log"\n        self.log_handle = log_path.open("ab", buffering=0)\n        command = [\n            "python", "-m", "vllm.entrypoints.openai.api_server",\n            "--model", str(model_path),\n            "--served-model-name", spec.served_name,\n            "--host", "127.0.0.1", "--port", str(spec.port),\n            "--max-model-len", str(spec.max_model_len),\n            "--gpu-memory-utilization", str(spec.gpu_memory_utilization),\n            "--tensor-parallel-size", "1", "--trust-remote-code",\n            "--enable-prefix-caching", *spec.extra_args,\n        ]\n        env = os.environ.copy()\n        env.setdefault("CUDA_VISIBLE_DEVICES", "0")\n        env.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")\n        env.setdefault("VLLM_BLOCKSCALE_FP8_GEMM_FLASHINFER", "0")\n        self.process = subprocess.Popen(\n            command, stdout=self.log_handle, stderr=subprocess.STDOUT,\n            start_new_session=True, env=env,\n        )\n        self.active = spec\n        self._wait_ready()\n\n    @staticmethod\n    def _resolve_model_path(root: Path) -> Path:\n        if (root / "config.json").exists():\n            return root\n        if root.exists():\n            candidates = sorted(root.glob("**/config.json"), key=lambda path: len(path.parts))\n            if candidates:\n                return candidates[0].parent\n        return root\n\n    def _wait_ready(self) -> None:\n        assert self.active is not None\n        deadline = time.monotonic() + self.startup_timeout\n        url = f"http://127.0.0.1:{self.active.port}/v1/models"\n        while time.monotonic() < deadline:\n            if self.process and self.process.poll() is not None:\n                raise RuntimeError(f"vLLM exited while loading {self.active.policy}")\n            try:\n                with urllib.request.urlopen(url, timeout=3) as response:\n                    payload = json.load(response)\n                if payload.get("data"):\n                    return\n            except Exception:\n                time.sleep(1)\n        raise TimeoutError(f"vLLM did not load {self.active.policy} in time")\n\n    def stop(self) -> None:\n        if self.process and self.process.poll() is None:\n            self.process.terminate()\n            try:\n                self.process.wait(timeout=90)\n            except subprocess.TimeoutExpired:\n                self.process.kill()\n                self.process.wait(timeout=30)\n        if self.log_handle is not None:\n            self.log_handle.close()\n        self.process = None\n        self.log_handle = None\n        self.active = None\n\n    def __enter__(self) -> "VllmRuntime":\n        return self\n\n    def __exit__(self, *_args) -> None:\n        self.stop()\n', 'online_learning.py': 'from __future__ import annotations\n\nfrom typing import Any\n\nimport numpy as np\n\n\nCHECKLIST_ITEMS = (\n    ("legal_controls", "Enumerate every currently legal control."),\n    ("controls_tested", "Test every legal control at least once when safe."),\n    ("causal_effect_found", "Identify a control that visibly changes the state."),\n    ("no_op_found", "Identify controls or contexts that produce no visible effect."),\n    ("progress_observed", "Observe direct level progress from a live action."),\n    ("repeatability_checked", "Repeat at least one control to test consistency."),\n    ("responsive_plan", "Prefer live-evidence actions over unsupported guesses."),\n)\n\n\ndef _frame(value: Any) -> list[list[int]] | None:\n    try:\n        array = np.asarray(value)\n        while array.ndim > 2 and array.shape[0] == 1:\n            array = array[0]\n    except (TypeError, ValueError):\n        return None\n    if array.ndim != 2:\n        return None\n    return array.astype(int).tolist()\n\n\ndef learn_transition(\n    memory: dict[str, Any], before: Any, action: Any, after: Any, *, max_recent: int = 32\n) -> None:\n    """Learn an action effect from one live transition in the current attempt."""\n    old = _frame(getattr(before, "frame", None))\n    new = _frame(getattr(after, "frame", None))\n    if old is None or new is None or len(old) != len(new):\n        return\n    if any(len(a) != len(b) for a, b in zip(old, new)):\n        return\n    changed = sum(a != b for old_row, new_row in zip(old, new) for a, b in zip(old_row, new_row))\n    action_name = getattr(action, "name", str(action))\n    effects = memory.setdefault("learned_action_effects", {})\n    stats = effects.setdefault(action_name, {"trials": 0, "changed_cells_total": 0, "progress": 0})\n    stats["trials"] += 1\n    stats["changed_cells_total"] += changed\n    progress = max(\n        0,\n        int(getattr(after, "levels_completed", 0))\n        - int(getattr(before, "levels_completed", 0)),\n    )\n    stats["progress"] += progress\n    recent = memory.setdefault("learned_transitions", [])\n    recent.append({"action": action_name, "changed_cells": changed, "progress": progress})\n    del recent[:-max_recent]\n    memory["learning_scope"] = "current_attempt_only"\n\n\ndef learning_checklist(memory: dict[str, Any], legal_actions: list[str]) -> dict[str, Any]:\n    """Return a game-general checklist completed solely from current-attempt evidence."""\n    observed_legal = memory.setdefault("observed_legal_controls", [])\n    for action in legal_actions:\n        if action not in observed_legal:\n            observed_legal.append(action)\n    effects = memory.get("learned_action_effects", {})\n    trials = {name: int(stats.get("trials", 0)) for name, stats in effects.items()}\n    tested = [name for name in observed_legal if trials.get(name, 0) > 0]\n    responsive = [\n        name for name, stats in effects.items()\n        if int(stats.get("changed_cells_total", 0)) > 0\n    ]\n    no_ops = [\n        name for name, stats in effects.items()\n        if int(stats.get("trials", 0)) > 0 and int(stats.get("changed_cells_total", 0)) == 0\n    ]\n    progress = [name for name, stats in effects.items() if int(stats.get("progress", 0)) > 0]\n    evidence = {\n        "legal_controls": bool(observed_legal),\n        "controls_tested": bool(observed_legal) and len(tested) == len(observed_legal),\n        "causal_effect_found": bool(responsive),\n        "no_op_found": bool(no_ops),\n        "progress_observed": bool(progress),\n        "repeatability_checked": any(count >= 2 for count in trials.values()),\n        "responsive_plan": bool(progress or responsive),\n    }\n    items = [\n        {"id": item_id, "requirement": requirement, "complete": evidence[item_id]}\n        for item_id, requirement in CHECKLIST_ITEMS\n    ]\n    checklist = {\n        "scope": "current_attempt_only",\n        "items": items,\n        "completed": sum(item["complete"] for item in items),\n        "total": len(items),\n        "tested_controls": tested,\n        "responsive_controls": responsive,\n        "progress_controls": progress,\n    }\n    memory["learning_checklist"] = checklist\n    return checklist\n\n\ndef action_plan(memory: dict[str, Any], legal_actions: list[str]) -> list[dict[str, Any]]:\n    """Rank experiments using evidence learned in this attempt only."""\n    effects = memory.get("learned_action_effects", {})\n    plan = []\n    for action in legal_actions:\n        stats = effects.get(action, {})\n        trials = int(stats.get("trials", 0))\n        changed = int(stats.get("changed_cells_total", 0))\n        progress = int(stats.get("progress", 0))\n        plan.append({\n            "action": action,\n            "trials": trials,\n            "mean_changed_cells": round(changed / trials, 2) if trials else None,\n            "progress": progress,\n            # Explore unseen controls first, then prefer demonstrated progress\n            # and observable causal effects. All terms are current-run evidence.\n            "priority": [int(trials == 0), progress, changed / trials if trials else 0, -trials],\n        })\n    plan.sort(key=lambda item: item["priority"], reverse=True)\n    return plan\n', 'phase_plan.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\n\n@dataclass(frozen=True)\nclass Phase:\n    model: str\n    direction: str\n    segment: str\n\n\ndef practice_phase_plan() -> tuple[Phase, ...]:\n    """Three checkpoint loads instead of four loads per individual game."""\n    return (\n        Phase("duck", "duck_to_public", "first"),\n        Phase("public", "duck_to_public", "second"),\n        Phase("public", "public_to_duck", "first"),\n        Phase("duck", "public_to_duck", "second"),\n    )\n\n\ndef competition_phase_plan(selected: dict[str, str]) -> tuple[Phase, ...]:\n    directions = set(selected.values())\n    phases: list[Phase] = []\n    if "duck_to_public" in directions:\n        phases.append(Phase("duck", "duck_to_public", "first"))\n    if "public_to_duck" in directions:\n        phases.append(Phase("public", "public_to_duck", "first"))\n    if "duck_to_public" in directions:\n        phases.append(Phase("public", "duck_to_public", "second"))\n    if "public_to_duck" in directions:\n        phases.append(Phase("duck", "public_to_duck", "second"))\n    return tuple(phases)\n', 'public_policies.py': 'from __future__ import annotations\n\nimport base64\nimport io\nimport json\nimport re\nimport os\nimport time\nimport urllib.request\nfrom typing import Any\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom arcengine import GameAction\n\nfrom .runner import PolicyContext, PolicyDecision\nfrom .online_learning import action_plan, learning_checklist\n\n\nPALETTE = np.array([\n    [0, 0, 0], [0, 116, 217], [255, 65, 54], [46, 204, 64],\n    [255, 220, 0], [170, 170, 170], [240, 18, 190], [255, 133, 27],\n    [127, 219, 255], [135, 12, 37], [57, 204, 204], [177, 13, 201],\n    [1, 255, 112], [133, 20, 75], [61, 153, 112], [221, 221, 221],\n], dtype=np.uint8)\n\nACTION_NAMES = {\n    "up": GameAction.ACTION1, "down": GameAction.ACTION2,\n    "left": GameAction.ACTION3, "right": GameAction.ACTION4,\n    "space": GameAction.ACTION5, "spacebar": GameAction.ACTION5,\n    "click": GameAction.ACTION6, "undo": GameAction.ACTION7,\n    "reset": GameAction.RESET,\n}\n\n\nclass PublicNotebookVisionPolicy:\n    """OpenAI-compatible adapter distilled from the two public notebooks."""\n\n    def __init__(self, profile: str, served_name: str, port: int = 8000) -> None:\n        if profile not in {"duck", "public"}:\n            raise ValueError("profile must be duck or public")\n        self.profile = profile\n        self.name = f"{profile}-public-notebook"\n        self.served_name = served_name\n        self.url = f"http://127.0.0.1:{port}/v1/chat/completions"\n        self.trace_path = Path(os.getenv("DUAL_TRACE_PATH", "artifacts/dual-model/full-trace.jsonl"))\n\n    def reset(self) -> None:\n        # All cognitive state lives in the per-attempt context/snapshot.\n        return None\n\n    def decide(self, observation: Any, context: PolicyContext) -> PolicyDecision:\n        image = self._image_data_url(getattr(observation, "frame", []))\n        prompt = self._prompt(observation, context)\n        body = {\n            "model": self.served_name,\n            "messages": [{"role": "user", "content": [\n                {"type": "text", "text": prompt},\n                {"type": "image_url", "image_url": {"url": image}},\n            ]}],\n            "temperature": 0.2 if self.profile == "duck" else 0.1,\n            "max_tokens": 1024,\n            # Qwen3 otherwise spends the entire token budget in\n            # reasoning_content and emits an empty final content field. The\n            # requested JSON still carries its concise, logged reasoning.\n            "chat_template_kwargs": {"enable_thinking": False},\n        }\n        request = urllib.request.Request(\n            self.url, data=json.dumps(body).encode(),\n            headers={"Content-Type": "application/json", "Authorization": "Bearer local"},\n        )\n        with urllib.request.urlopen(request, timeout=400) as response:\n            payload = json.load(response)\n        text = self._message_text(payload["choices"][0]["message"])\n        decision = self._parse(text, observation)\n        context.memory["last_policy"] = self.profile\n        context.memory["last_reasoning"] = decision.reasoning[-1500:]\n        self._trace(context, prompt, text, decision)\n        return decision\n\n    @staticmethod\n    def _message_text(message: dict[str, Any]) -> str:\n        """Extract OpenAI/vLLM text across plain, multimodal, and reasoning replies."""\n        parts: list[str] = []\n        reasoning = message.get("reasoning_content")\n        if isinstance(reasoning, str) and reasoning.strip():\n            parts.append(reasoning)\n        content = message.get("content")\n        if isinstance(content, str) and content.strip():\n            parts.append(content)\n        elif isinstance(content, list):\n            for block in content:\n                if isinstance(block, str) and block.strip():\n                    parts.append(block)\n                elif isinstance(block, dict):\n                    text = block.get("text")\n                    if isinstance(text, str) and text.strip():\n                        parts.append(text)\n        for tool_call in message.get("tool_calls") or []:\n            arguments = (tool_call.get("function") or {}).get("arguments")\n            if isinstance(arguments, str) and arguments.strip():\n                parts.append(arguments)\n        return "\\n".join(parts)\n\n    def _trace(self, context: PolicyContext, prompt: str, response: str, decision: PolicyDecision) -> None:\n        record = {\n            "timestamp": time.time(), "game_id": context.game_id,\n            "direction": context.direction, "policy": self.profile,\n            "action_index": context.action_index, "handoff_action": context.handoff_action,\n            "prompt": prompt, "model_response": response,\n            "move": decision.action.name, "move_data": decision.data,\n            "reasoning": decision.reasoning, "world_model": context.memory,\n        }\n        self.trace_path.parent.mkdir(parents=True, exist_ok=True)\n        with self.trace_path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n        print(\n            "MODEL_TRACE " + json.dumps(record, ensure_ascii=False)\n            if os.getenv("DUAL_ECHO_FULL_TRACE", "0").lower() in {"1", "true", "yes"}\n            else f"MOVE game={context.game_id} step={context.action_index} "\n                 f"policy={self.profile} action={decision.action.name} reasoning={decision.reasoning}"\n        )\n\n    def _prompt(self, observation: Any, context: PolicyContext) -> str:\n        legal = []\n        for item in getattr(observation, "available_actions", None) or []:\n            try:\n                action = item if isinstance(item, GameAction) else GameAction.from_id(int(item))\n                legal.append(action.name)\n            except (TypeError, ValueError):\n                legal.append(str(item))\n        checklist = learning_checklist(context.memory, legal)\n        memory = json.dumps(context.memory, ensure_ascii=True)[:4000]\n        transition = self._transition_summary(context.observations)\n        experiments = json.dumps(action_plan(context.memory, legal), ensure_ascii=True)\n        shared = (\n            f"ARC-AGI-3 game, action {context.action_index}, direction {context.direction}. "\n            f"Legal engine actions: {legal}. Latest transition: {transition}. "\n            f"Current-attempt memory: {memory}. "\n            f"Evidence-ranked legal experiments: {experiments}. "\n            f"Mandatory learning checklist: {json.dumps(checklist, ensure_ascii=True)}. "\n            "Work on pending checklist items before committing to a plan; treat an item as complete "\n            "only when the current attempt supplied its evidence. "\n            "There are no task-specific priors. Infer only from this attempt\'s visual evidence. "\n            "Return one JSON object: {\\"action\\":\\"up|down|left|right|space|click|undo|reset\\","\n            "\\"x\\":0,\\"y\\":0,\\"reasoning\\":\\"brief evidence\\"}. Include x/y only for click."\n        )\n        if self.profile == "duck":\n            return shared + (\n                " Tufa/Duck public policy: build a compact causal world model; compare the latest "\n                "transition, identify controllable components, prefer reversible experiments, and "\n                "commit to a short reliable plan only when supported by evidence."\n            )\n        return shared + (\n            " Public 0.86 policy: use chronological visual context, avoid repeated states, prefer "\n            "purposeful novelty, reject implausible clicks, and choose one conservative action."\n        )\n\n    @staticmethod\n    def _transition_summary(observations: list[Any]) -> str:\n        if len(observations) < 2:\n            return "initial frame; no action transition yet"\n        before = np.asarray(getattr(observations[-2], "frame", []))\n        after = np.asarray(getattr(observations[-1], "frame", []))\n        if before.shape != after.shape or not before.size:\n            return f"frame shape changed from {before.shape} to {after.shape}"\n        changed = int(np.count_nonzero(before != after))\n        return (\n            f"changed_cells={changed}, shape={after.shape}, "\n            f"levels_completed={getattr(observations[-1], \'levels_completed\', 0)}"\n        )\n\n    @staticmethod\n    def _parse(text: str, observation: Any) -> PolicyDecision:\n        parsed: dict[str, Any] = {}\n        decoder = json.JSONDecoder()\n        for match in re.finditer(r"\\{", text):\n            try:\n                candidate, _ = decoder.raw_decode(text[match.start():])\n            except json.JSONDecodeError:\n                continue\n            if isinstance(candidate, dict) and "action" in candidate:\n                parsed = candidate\n                break\n        name = str(parsed.get("action", "")).lower().strip()\n        action = ACTION_NAMES.get(name)\n        available = [\n            item if isinstance(item, GameAction) else GameAction.from_id(int(item))\n            for item in (getattr(observation, "available_actions", None) or [])\n        ]\n        if action is None or (available and action not in available):\n            action = next((item for item in available if item != GameAction.RESET), GameAction.RESET)\n        data = {}\n        if action.is_complex():\n            data = {"x": int(parsed.get("x", 32)), "y": int(parsed.get("y", 32))}\n        return PolicyDecision(action, data, str(parsed.get("reasoning", text))[:2000])\n\n    @staticmethod\n    def _image_data_url(frame: Any) -> str:\n        array = np.asarray(frame)\n        while array.ndim > 2:\n            array = array[-1]\n        if array.ndim != 2:\n            array = np.zeros((64, 64), dtype=np.uint8)\n        rgb = PALETTE[np.clip(array.astype(int), 0, len(PALETTE) - 1)]\n        image = Image.fromarray(rgb, "RGB").resize(\n            (rgb.shape[1] * 8, rgb.shape[0] * 8), Image.Resampling.NEAREST\n        )\n        buffer = io.BytesIO()\n        image.save(buffer, format="PNG")\n        return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()\n\n\ndef create_public_dual_policies() -> tuple[PublicNotebookVisionPolicy, PublicNotebookVisionPolicy]:\n    return (\n        PublicNotebookVisionPolicy("duck", "duck-qwen36-27b"),\n        PublicNotebookVisionPolicy("public", "public-gemma4-31b"),\n    )\n', 'runner.py': 'from __future__ import annotations\n\nimport json\nimport random\nimport time\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nfrom arc_agi import Arcade, OperationMode\nfrom arcengine import GameAction, GameState\n\nfrom .online_learning import learn_transition\n\n\n@dataclass(frozen=True)\nclass PolicyDecision:\n    action: GameAction\n    data: dict[str, int] = field(default_factory=dict)\n    reasoning: str = ""\n\n\n@dataclass\nclass PolicyContext:\n    game_id: str\n    direction: str\n    active_policy: str\n    action_index: int\n    handoff_action: int\n    observations: list[Any]\n    memory: dict[str, Any]\n\n\nclass Policy(Protocol):\n    name: str\n\n    def decide(self, observation: Any, context: PolicyContext) -> PolicyDecision: ...\n\n    def reset(self) -> None: ...\n\n\nclass CyclingPolicy:\n    """API smoke-test policy; deliberately not a leaderboard solver."""\n\n    def __init__(self, name: str, seed: int = 0) -> None:\n        self.name = name\n        self._seed = seed\n        self._rng = random.Random(seed)\n\n    def reset(self) -> None:\n        self._rng = random.Random(self._seed)\n\n    def decide(self, observation: Any, context: PolicyContext) -> PolicyDecision:\n        available = [\n            item if isinstance(item, GameAction) else GameAction.from_id(int(item))\n            for item in (getattr(observation, "available_actions", None) or [])\n        ]\n        available = [action for action in available if action != GameAction.RESET]\n        if not available:\n            return PolicyDecision(GameAction.RESET, reasoning=f"{self.name}: reset required")\n        action = available[self._rng.randrange(len(available))]\n        data: dict[str, int] = {}\n        if action.is_complex():\n            frame = getattr(observation, "frame", None)\n            height = len(frame) if frame is not None else 64\n            width = len(frame[0]) if height and frame is not None else 64\n            data = {"x": max(0, width // 2), "y": max(0, height // 2)}\n        return PolicyDecision(action, data, f"{self.name}: official Arcade smoke action")\n\n\n@dataclass(frozen=True)\nclass HarnessConfig:\n    game_ids: tuple[str, ...]\n    attempts: int = 2\n    max_actions: int | None = 200\n    handoff_action: int = 100\n    qualification_levels: int = 1\n    max_practice_rounds: int = 3\n    global_time_limit_seconds: float = 9 * 60 * 60\n    shutdown_reserve_seconds: float = 20 * 60\n    preferred_direction: str = "duck_to_public"\n    source_url: str | None = None\n    tags: tuple[str, ...] = ("arc3-handoff", "test-harness")\n    save_recordings: bool = True\n    output: Path = Path("artifacts/arcade-results.json")\n\n    def validate(self) -> None:\n        if not self.game_ids:\n            raise ValueError("at least one game id is required")\n        if self.attempts < 1:\n            raise ValueError("attempts must be positive")\n        if self.max_actions is not None and self.max_actions < 1:\n            raise ValueError("max_actions must be positive")\n        if self.handoff_action < 1:\n            raise ValueError("handoff_action must be positive")\n        if self.max_actions is not None and self.handoff_action >= self.max_actions:\n            raise ValueError("handoff_action must be below max_actions when a test cap is used")\n        if self.qualification_levels < 1 or self.max_practice_rounds < 1:\n            raise ValueError("practice qualification settings must be positive")\n        if self.global_time_limit_seconds <= self.shutdown_reserve_seconds:\n            raise ValueError("global time limit must exceed the shutdown reserve")\n        if self.preferred_direction not in {"duck_to_public", "public_to_duck"}:\n            raise ValueError("invalid preferred direction")\n\n\n@dataclass(frozen=True)\nclass RunResult:\n    game_id: str\n    attempt: int\n    direction: str\n    actions: int\n    levels_completed: int\n    final_state: str\n    elapsed_seconds: float\n    handoff_reached: bool\n    score: float = 0.0\n\n\nclass ArcadeHarness:\n    def __init__(\n        self,\n        config: HarnessConfig,\n        *,\n        practice_arcade: Arcade | None = None,\n        competition_arcade: Arcade | None = None,\n        duck_policy: Policy | None = None,\n        public_policy: Policy | None = None,\n    ) -> None:\n        config.validate()\n        self.config = config\n        self.practice_arcade = practice_arcade or Arcade(operation_mode=OperationMode.ONLINE)\n        self.competition_arcade = competition_arcade or Arcade(operation_mode=OperationMode.COMPETITION)\n        self.policies: dict[str, Policy] = {\n            "duck": duck_policy or CyclingPolicy("duck-smoke", seed=918),\n            "public": public_policy or CyclingPolicy("public-smoke", seed=860),\n        }\n\n    def _reset_policies(self) -> None:\n        for policy in self.policies.values():\n            reset = getattr(policy, "reset", None)\n            if callable(reset):\n                reset()\n\n    def discover(self) -> list[dict[str, Any]]:\n        return [\n            {\n                "game_id": item.game_id,\n                "title": getattr(item, "title", ""),\n                "tags": list(getattr(item, "tags", None) or []),\n            }\n            for item in self.practice_arcade.get_environments()\n        ]\n\n    def _directions(self) -> list[str]:\n        reverse = (\n            "public_to_duck"\n            if self.config.preferred_direction == "duck_to_public"\n            else "duck_to_public"\n        )\n        return [self.config.preferred_direction, reverse]\n\n    def run_practice(self) -> dict[str, Any]:\n        """Run both directions online, close the practice card, and pick winners."""\n        card_id = self.practice_arcade.create_scorecard(\n            source_url=self.config.source_url,\n            tags=list(self.config.tags),\n            opaque={\n                "phase": "practice",\n                "attempts": self.config.attempts,\n                "handoff_action": self.config.handoff_action,\n                "preferred_direction": self.config.preferred_direction,\n            },\n        )\n        results: list[RunResult] = []\n        final_scorecard = None\n        try:\n            directions = self._directions()\n            for game_id in self.config.game_ids:\n                for attempt in range(self.config.attempts):\n                    # No-priors boundary: do not carry policy state across attempts.\n                    self._reset_policies()\n                    results.append(\n                        self._play_one(self.practice_arcade, game_id, attempt, directions[attempt % 2], card_id)\n                    )\n        finally:\n            final_scorecard = self.practice_arcade.close_scorecard(scorecard_id=card_id)\n\n        selected = self._select_directions(results)\n\n        payload = {\n            "scorecard_id": card_id,\n            "phase": "practice",\n            "scorecard": (\n                final_scorecard.model_dump(mode="json") if final_scorecard is not None else None\n            ),\n            "runs": [asdict(result) for result in results],\n            "selected_directions": selected,\n        }\n        self.config.output.parent.mkdir(parents=True, exist_ok=True)\n        self.config.output.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n        return payload\n\n    @staticmethod\n    def _select_directions(results: list[RunResult]) -> dict[str, str]:\n        """Select max levels; retain the first attempt on a tie like Arcade."""\n        best: dict[str, RunResult] = {}\n        for result in results:\n            current = best.get(result.game_id)\n            if current is None or result.levels_completed > current.levels_completed:\n                best[result.game_id] = result\n        return {game_id: result.direction for game_id, result in best.items()}\n\n    def run_competition(self, selected: dict[str, str]) -> dict[str, Any]:\n        """Make each competition environment once using its practice winner."""\n        missing = sorted(set(self.config.game_ids) - set(selected))\n        if missing:\n            raise ValueError(f"practice selection missing games: {missing}")\n        card_id = self.competition_arcade.create_scorecard(\n            source_url=self.config.source_url,\n            tags=[*self.config.tags, "competition"],\n            opaque={"phase": "competition", "practice_winners": selected},\n        )\n        results: list[RunResult] = []\n        final_scorecard = None\n        # Practice frames, reflections, plans, and RNG state cannot enter the\n        # scored phase. Only the selected direction label crosses this boundary.\n        self._reset_policies()\n        try:\n            for game_id in self.config.game_ids:\n                self._reset_policies()\n                results.append(self._play_one(\n                    self.competition_arcade, game_id, 0, selected[game_id], card_id\n                ))\n        finally:\n            final_scorecard = self.competition_arcade.close_scorecard(scorecard_id=card_id)\n        return {\n            "scorecard_id": card_id,\n            "phase": "competition",\n            "scorecard": final_scorecard.model_dump(mode="json") if final_scorecard else None,\n            "runs": [asdict(result) for result in results],\n            "selected_directions": selected,\n        }\n\n    def run_two_stage(self) -> dict[str, Any]:\n        practice = self.run_practice()\n        competition = self.run_competition(practice["selected_directions"])\n        payload = {"practice": practice, "competition": competition}\n        combined = self.config.output.with_name(\n            f"{self.config.output.stem}-two-stage{self.config.output.suffix}"\n        )\n        combined.parent.mkdir(parents=True, exist_ok=True)\n        combined.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n        return payload\n\n    def _play_one(self, arcade: Arcade, game_id: str, attempt: int, direction: str, card_id: str) -> RunResult:\n        env = arcade.make(\n            game_id,\n            scorecard_id=card_id,\n            save_recording=self.config.save_recordings,\n            include_frame_data=True,\n        )\n        if env is None:\n            raise RuntimeError(f"Arcade could not create environment {game_id}")\n        observation = env.observation_space\n        observations = [observation]\n        # Fresh per-attempt memory is the core no-priors invariant.\n        shared_memory: dict[str, Any] = {}\n        started = time.monotonic()\n        actions_taken = 0\n        action_index = 0\n        while self.config.max_actions is None or action_index < self.config.max_actions:\n            if getattr(observation, "state", None) == GameState.WIN:\n                break\n            if getattr(observation, "state", None) in {GameState.NOT_PLAYED, GameState.GAME_OVER}:\n                decision = PolicyDecision(GameAction.RESET, reasoning="Arcade state requires reset")\n            else:\n                first, second = direction.split("_to_")\n                active = second if action_index >= self.config.handoff_action else first\n                context = PolicyContext(\n                    game_id=game_id,\n                    direction=direction,\n                    active_policy=active,\n                    action_index=action_index,\n                    handoff_action=self.config.handoff_action,\n                    observations=observations,\n                    memory=shared_memory,\n                )\n                decision = self.policies[active].decide(observation, context)\n            reasoning = {"text": decision.reasoning} if decision.reasoning else None\n            observation = env.step(decision.action, data=decision.data, reasoning=reasoning)\n            if observation is None:\n                raise RuntimeError(f"Arcade returned no observation for {game_id}")\n            learn_transition(shared_memory, observations[-1], decision.action, observation)\n            observations.append(observation)\n            actions_taken += 1\n            action_index += 1\n\n        return RunResult(\n            game_id=game_id,\n            attempt=attempt,\n            direction=direction,\n            actions=actions_taken,\n            levels_completed=int(getattr(observation, "levels_completed", 0)),\n            final_state=getattr(getattr(observation, "state", None), "name", "UNKNOWN"),\n            elapsed_seconds=round(time.monotonic() - started, 3),\n            handoff_reached=actions_taken > self.config.handoff_action,\n        )\n', 'snapshot.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\n\nSCHEMA_VERSION = 1\n\n\ndef _json_safe(value: Any) -> Any:\n    """Recursively normalize NumPy-backed Arcade state for stable JSON."""\n    if isinstance(value, dict):\n        return {str(key): _json_safe(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [_json_safe(item) for item in value]\n    if hasattr(value, "tolist"):\n        return _json_safe(value.tolist())\n    if hasattr(value, "item"):\n        return _json_safe(value.item())\n    return value\n\n\n@dataclass\nclass AttemptSnapshot:\n    """No-priors, within-attempt state transferred at a model handoff."""\n\n    run_id: str\n    game_id: str\n    attempt_id: str\n    direction: str\n    action_index: int\n    handoff_action: int\n    levels_completed: int = 0\n    current_frame: list[list[int]] = field(default_factory=list)\n    recent_transitions: list[dict[str, Any]] = field(default_factory=list)\n    world_model: dict[str, Any] = field(default_factory=dict)\n    failed_actions: dict[str, list[str]] = field(default_factory=dict)\n    schema_version: int = SCHEMA_VERSION\n\n    def validate(self) -> None:\n        if self.schema_version != SCHEMA_VERSION:\n            raise ValueError(f"unsupported snapshot schema {self.schema_version}")\n        if self.direction not in {"duck_to_public", "public_to_duck"}:\n            raise ValueError("invalid handoff direction")\n        if not self.run_id or not self.game_id or not self.attempt_id:\n            raise ValueError("snapshot identity fields cannot be empty")\n        if self.action_index < 0 or self.handoff_action < 1:\n            raise ValueError("invalid action counters")\n\n    def digest(self) -> str:\n        payload = json.dumps(_json_safe(asdict(self)), sort_keys=True, separators=(",", ":"))\n        return hashlib.sha256(payload.encode()).hexdigest()\n\n\nclass SnapshotStore:\n    """Atomic snapshot persistence with strict run/attempt isolation."""\n\n    def __init__(self, root: Path) -> None:\n        self.root = root\n\n    @staticmethod\n    def _safe(value: str) -> str:\n        if not value or any(char not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for char in value):\n            raise ValueError(f"unsafe snapshot identifier: {value!r}")\n        return value\n\n    def path_for(self, snapshot: AttemptSnapshot) -> Path:\n        return (\n            self.root\n            / self._safe(snapshot.run_id)\n            / self._safe(snapshot.game_id)\n            / f"{self._safe(snapshot.attempt_id)}.json"\n        )\n\n    def save(self, snapshot: AttemptSnapshot) -> Path:\n        snapshot.validate()\n        path = self.path_for(snapshot)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        payload = {"snapshot": _json_safe(asdict(snapshot)), "sha256": snapshot.digest()}\n        temporary = path.with_suffix(".tmp")\n        temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n        temporary.replace(path)\n        return path\n\n    def load(self, path: Path, *, run_id: str, attempt_id: str) -> AttemptSnapshot:\n        payload = json.loads(path.read_text(encoding="utf-8"))\n        snapshot = AttemptSnapshot(**payload["snapshot"])\n        snapshot.validate()\n        if snapshot.run_id != run_id or snapshot.attempt_id != attempt_id:\n            raise ValueError("no-priors boundary violation: snapshot belongs to another attempt")\n        if payload.get("sha256") != snapshot.digest():\n            raise ValueError("snapshot integrity check failed")\n        return snapshot\n'}
package_root = Path('/kaggle/working/dual_model_package/arc3_harness')
package_root.mkdir(parents=True, exist_ok=True)
for name, source in PACKAGE.items():
    (package_root / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(package_root.parent))
print(f'Embedded {len(PACKAGE)} dual-model modules at {package_root}')


In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path

import torch

def validate_vision_checkpoint(label, path):
    root = Path(path)
    config_path = root / 'config.json'
    if not config_path.exists():
        raise FileNotFoundError(f'{label}: config.json missing at {root}')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    architectures = config.get('architectures') or []
    config_text = json.dumps(config).lower()
    multimodal = 'vision' in config_text or any(
        token in architecture.lower()
        for architecture in architectures
        for token in ('conditionalgeneration', 'vl', 'vision')
    )
    if not architectures or not multimodal:
        raise RuntimeError(f'{label}: checkpoint is not a verified vision-generation model')
    weight_files = list(root.glob('*.safetensors')) + list(root.glob('*.bin'))
    index_files = list(root.glob('*.index.json'))
    if not weight_files and not index_files:
        raise FileNotFoundError(f'{label}: model weights are missing at {root}')
    print(label, {'path': str(root), 'architectures': architectures, 'weight_files': len(weight_files)})

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for the 27B/31B vLLM checkpoints')
print('GPU:', torch.cuda.get_device_name(0))
print('vLLM:', importlib.metadata.version('vllm'))
validate_vision_checkpoint('duck', os.environ['ARC3_DUCK_MODEL_PATH'])
validate_vision_checkpoint('public', os.environ['ARC3_PUBLIC_MODEL_PATH'])

# Online training state is deliberately empty. The harness creates isolated
# in-memory action models after each fresh Arcade environment is opened.
training_root = Path('/kaggle/working/online-training')
training_root.mkdir(parents=True, exist_ok=True)
(training_root / 'manifest.json').write_text(json.dumps({
    'mode': 'online_per_attempt',
    'initial_state': 'empty',
    'loads_recorded_moves': False,
    'loads_offline_world_model': False,
}, indent=2), encoding='utf-8')
print('No-priors online training workspace ready:', training_root)


In [ ]:
import json
import os
from pathlib import Path

from arc_agi import Arcade, OperationMode
from arc3_harness.dual_runner import DualModelArcadeHarness
from arc3_harness.model_runtime import VllmRuntime
from arc3_harness.public_policies import create_public_dual_policies
from arc3_harness.runner import HarnessConfig
from arc3_harness.snapshot import SnapshotStore

true_submission = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
api_key = os.getenv('ARC_API_KEY') or 'test-key-123'
base_url = os.getenv('ARC_BASE_URL', 'http://gateway:8001')
environment_candidates = [
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files'),
    Path('/kaggle/input/arc-prize-2026-arc-agi-3/environment_files'),
]
environment_dir = next((path for path in environment_candidates if path.exists()), None)
if environment_dir is None:
    raise FileNotFoundError('official local practice environment_files are missing')

# Practice is official Arcade in OFFLINE mode. It can legally create both fresh
# attempts without touching or contaminating the competition scorecard.
practice_arcade = Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=str(environment_dir),
    recordings_dir='/kaggle/working/practice-recordings',
)

practice_ids = {game.game_id for game in practice_arcade.get_environments()}
if true_submission:
    competition_arcade = Arcade(
        arc_api_key=api_key, arc_base_url=base_url,
        operation_mode=OperationMode.COMPETITION,
        recordings_dir='/kaggle/working/competition-recordings',
    )
    game_ids = tuple(game.game_id for game in competition_arcade.get_environments())
    if not game_ids:
        raise RuntimeError('competition gateway returned no private games')
    print(f'PRIVATE_COMPETITION_GAMES count={len(game_ids)}')
else:
    # A normal commit has no competition gateway. Give all nine GPU hours to
    # the complete 25-game public suite instead.
    competition_arcade = practice_arcade
    game_ids = tuple(sorted(practice_ids))
    if len(game_ids) != 25:
        raise RuntimeError(f'expected all 25 public games, found {len(game_ids)}')
    print(f'PUBLIC_FULL_SUITE count={len(game_ids)}')

duck_policy, public_policy = create_public_dual_policies()
config = HarnessConfig(
    game_ids=game_ids, attempts=2,
    # No move cap: action 100 is only the policy handoff, never a stop point.
    max_actions=None,
    handoff_action=int(os.getenv('DUAL_HANDOFF_ACTION', '100')),
    qualification_levels=int(os.getenv('DUAL_QUALIFICATION_LEVELS', '1')),
    max_practice_rounds=int(os.getenv('DUAL_MAX_PRACTICE_ROUNDS', '3')),
    global_time_limit_seconds=9 * 60 * 60,
    shutdown_reserve_seconds=20 * 60,
    preferred_direction=os.getenv('DUAL_PREFERRED_DIRECTION', 'duck_to_public'),
    source_url='https://github.com/BlockChain-BailBonds/arc3-milestone-handoff',
    output=Path('/kaggle/working/dual-model-results.json'),
)
harness = DualModelArcadeHarness(
    config,
    practice_arcade=practice_arcade,
    competition_arcade=competition_arcade,
    runtime=VllmRuntime(log_dir=Path('/kaggle/working/vllm-logs')),
    duck_policy=duck_policy,
    public_policy=public_policy,
    snapshot_store=SnapshotStore(Path('/kaggle/working/handoff-snapshots')),
    practice_game_ids=practice_ids,
)
if true_submission:
    # A separate nine-hour budget is devoted to every unseen gateway game.
    results = {'competition': harness.run_private_competition()}
    print('FINAL_TRUE_COMPETITION_SCORECARD')
    print(json.dumps(results['competition'], indent=2))
else:
    results = {'practice': harness.run_public_suite()}
    Path('/kaggle/working/public-25-results.json').write_text(
        json.dumps(results['practice'], indent=2), encoding='utf-8'
    )
    print('FINAL_PUBLIC_25_SCORECARD')
    print(json.dumps(results['practice'], indent=2))


In [ ]:
from pathlib import Path

submission = Path('/kaggle/working/submission.parquet')
trace = Path('/kaggle/working/dual-model-full-trace.jsonl')
results = Path('/kaggle/working/dual-model-results-dual-model.json')
true_submission = bool(__import__('os').getenv('KAGGLE_IS_COMPETITION_RERUN'))
if true_submission and not submission.exists():
    raise RuntimeError('official gateway did not emit submission.parquet')
if not trace.exists() or trace.stat().st_size == 0:
    raise RuntimeError('full model thought/plan/move trace is missing')
if true_submission:
    print(f'COMPETITION_SUBMISSION_READY bytes={submission.stat().st_size}')
else:
    public_results = Path('/kaggle/working/public-25-results.json')
    if not public_results.exists():
        raise RuntimeError('full public-suite results are missing')
    print(f'PUBLIC_25_RESULTS_READY bytes={public_results.stat().st_size}')
print(f'FULL_TRACE_READY bytes={trace.stat().st_size}')
